# Inside a Language Model

Everything a language model does to your text happens in four moves: **cut it into tokens**, **turn tokens into vectors**, **let the vectors look at each other**, and **turn the last vector back into a choice of next token**. This lab builds all four by hand in numpy, on arrays small enough that you can read every number — and then costs out the one optimisation that makes generation practical, the **KV cache**.

There is no `torch` here, and that is deliberate. The browser kernel has numpy and matplotlib and nothing else, which forces us to write the arithmetic out. Attention is nine lines of numpy; the reason it looks mysterious in a paper is notation, not complexity.

**How to use this notebook:** run cells top to bottom (`Shift+Enter`); later sections reuse earlier variables. Each section ends with a micro-exercise whose scaffold runs as-is — fill it in now or on a second pass. Companion reading: Chapters 26 and 27.

## 1. Tokenization: growing a vocabulary with BPE

A model cannot read characters and it cannot afford one entry per word (spelling mistakes, new words, and every language at once would blow the table up). **Byte-pair encoding** (BPE) splits the difference: start with single characters, then repeatedly glue together whichever *adjacent pair* is most frequent, and keep the list of glue operations. Common words end up as one token; rare words break into reusable pieces.

The trainer below is the real algorithm — GPT-2, Llama and Mistral tokenizers are trained by this loop. What is toy is the scale: forty-odd words and twenty merges instead of hundreds of gigabytes and 50,000 merges.

In [ ]:
from collections import Counter

CORPUS = """
a happy dog and a happy cat share the happiness of a walk
kindness and happiness are learned slowly
an unhappy cat is an unkind cat and an unkind cat is unhappy
unkindness and sadness follow unkind acts
the sadness of an unhappy dog is real sadness
kindness undoes unkindness and happiness undoes sadness
"""

def symbols(word):
    """A word as a tuple of symbols; '_' marks the end of the word."""
    return tuple(list(word) + ["_"])

def pair_counts(vocab):
    counts = Counter()
    for syms, freq in vocab.items():
        for pair in zip(syms, syms[1:]):        # every adjacent pair
            counts[pair] += freq
    return counts

def merge_pair(vocab, pair):
    """Replace every occurrence of `pair` with the glued-together symbol."""
    out = {}
    for syms, freq in vocab.items():
        new, i = [], 0
        while i < len(syms):
            if i + 1 < len(syms) and (syms[i], syms[i + 1]) == pair:
                new.append(syms[i] + syms[i + 1])
                i += 2
            else:
                new.append(syms[i])
                i += 1
        out[tuple(new)] = freq
    return out

def train_bpe(text, n_merges):
    vocab = {symbols(w): f for w, f in Counter(text.lower().split()).items()}
    merges = []
    for _ in range(n_merges):
        counts = pair_counts(vocab)
        if not counts:
            break
        # most frequent pair; ties broken alphabetically so runs are repeatable
        pair = min(counts.items(), key=lambda kv: (-kv[1], kv[0]))[0]
        merges.append((pair, counts[pair]))
        vocab = merge_pair(vocab, pair)
    return merges

merges = train_bpe(CORPUS, 20)
print(f"{len(merges)} merges learned from {len(CORPUS.split())} words\n")
for step, (pair, count) in enumerate(merges, start=1):
    print(f"{step:>2}. {pair[0] + ' + ' + pair[1]:<16} -> {pair[0] + pair[1]:<10} (seen {count}x)")

Read that list top to bottom and you are watching a vocabulary being born. Early merges glue `s` to the end-of-word marker, because plural and possessive endings are everywhere. A few steps later the tokenizer owns the whole suffix `ness_` as one symbol, then the stem `happ`, then the negation prefix `un`. Nobody told it about English morphology — those are simply the pairs that paid off.

To tokenize a *new* word we replay the merges in the order they were learned, always taking the earliest-learned merge still available.

In [ ]:
rank = {pair: i for i, (pair, _count) in enumerate(merges)}

def encode_word(word):
    syms = list(symbols(word))
    while True:
        options = [(rank[p], i) for i, p in enumerate(zip(syms, syms[1:])) if p in rank]
        if not options:
            return syms
        _, i = min(options)                       # earliest-learned merge wins
        syms[i:i + 2] = [syms[i] + syms[i + 1]]

def encode(text):
    return [t for w in text.lower().split() for t in encode_word(w)]

def decode(tokens):
    return "".join(tokens).replace("_", " ").strip()

base = {c for w in CORPUS.lower().split() for c in symbols(w)}
token_list = sorted(base | {a + b for (a, b), _c in merges})
token_id = {t: i for i, t in enumerate(token_list)}
print("vocabulary size:", len(token_list))

for word in ["happy", "happiness", "kindness", "unhappiness", "unkindly"]:
    print(f"{word:<13} -> {encode_word(word)}")

trip = "an unkind cat"
print("\ntokens:", encode(trip))
print("IDs   :", [token_id[t] for t in encode(trip)])
print("decode:", repr(decode(encode(trip))), "  round-trip ok:", decode(encode(trip)) == trip)

Look at `unhappiness`. That word never appears in the corpus, and the tokenizer does not blink: prefix, stem, joint, suffix. A word-level tokenizer would emit `<unk>` and destroy the meaning; a character tokenizer would emit eleven tokens. This is the whole reason BPE won.

### Micro-exercise: tokens per word

Write `tokens_per_word(text)` that returns the average number of tokens `encode_word` produces per whitespace word. Try it on a sentence made of corpus words and on one made of words the corpus never saw — the ratio is exactly what makes API bills for unusual text higher than you expect.

In [ ]:
def tokens_per_word(text):
    words = text.lower().split()
    # your code here: total tokens across all words, divided by len(words)
    return 0.0

# Uncomment to test:
# print(tokens_per_word("a happy cat and a happy dog"))     # familiar words
# print(tokens_per_word("photosynthesis quarterly rhythms"))  # unfamiliar words

## 2. Embeddings: meaning becomes geometry

Token IDs are useless arithmetic — token 40 is not "twice" token 20. So the very first layer of the model is a lookup table, the **embedding matrix**: one row of numbers per vocabulary entry. Getting a token's vector is literally indexing a row.

Below the four columns have been given human-readable meanings so you can see what is happening. In a real model these columns are learned from data and nobody labels them.

In [ ]:
import numpy as np

vocab = ["the", "cat", "chased", "a", "kitten"]
#                    animal  action  determiner  small
E = np.array([[0.0,   0.0,    1.0,       0.0],   # 0 the
              [0.9,   0.0,    0.0,       0.3],   # 1 cat
              [0.0,   1.0,    0.0,       0.0],   # 2 chased
              [0.0,   0.0,    0.9,       0.0],   # 3 a
              [0.9,   0.0,    0.0,       0.9]])  # 4 kitten

token_ids = [0, 1, 2, 3, 4]           # "the cat chased a kitten"
X = E[token_ids]                      # <- the whole embedding step: row lookup
print("embedding matrix:", E.shape, "(vocab_size, d_model)")
print("sequence        :", X.shape, "(n_tokens, d_model)")
print("vector for 'kitten':", E[4])

`X` is now a $5 \times 4$ block of numbers, and everything from here on is arithmetic on that block.

Two vectors mean similar things when they point in similar directions, which is measured by **cosine similarity**:

$$
\cos(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\lVert \mathbf{a} \rVert \, \lVert \mathbf{b} \rVert}
$$

The numerator is the **dot product** — multiply matching slots, add them up. The denominator divides out length, leaving only direction.

In [ ]:
def cosine(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

for w1, w2 in [("cat", "kitten"), ("cat", "chased"), ("the", "a"), ("kitten", "a")]:
    a, b = E[vocab.index(w1)], E[vocab.index(w2)]
    print(f"cos({w1:>6}, {w2:<7}) = {cosine(a, b):+.3f}")

# The dot product IS alignment, once lengths are fixed:
a = np.array([1.0, 0.0])
print()
for degrees in [0, 45, 90, 135, 180]:
    t = np.radians(degrees)
    b = np.array([np.cos(t), np.sin(t)])       # unit vector at this angle
    print(f"angle {degrees:>3} deg   a . b = {a @ b:+.2f}")

`cat` and `kitten` score high because they share the *animal* direction. `cat` and `chased` score exactly zero: their vectors have no dimension in common. `the` and `a` score exactly 1 — as far as this tiny model is concerned they are interchangeable. Meaning has become geometry, and "related" has become "points the same way".

The angle sweep is the mechanism underneath: same direction gives $+1$, perpendicular gives $0$, opposite gives $-1$. When a model asks "how much should token 3 care about token 1?", it computes a dot product. That is the entire scoring mechanism.

## 3. Attention, one step at a time

Here is the formula, from *Attention Is All You Need* (Vaswani et al., 2017):

$$
\operatorname{Attention}(Q, K, V) = \operatorname{softmax}\!\left(\frac{QK^{\top}}{\sqrt{d_k}}\right) V
$$

In one sentence: **every token builds a query saying what it is looking for, every token builds a key advertising what it offers, we match queries against keys with dot products, turn the matches into weights, and each token's output becomes a weighted blend of the other tokens' values.**

Step 1 is three matrix multiplies. Each token's embedding is projected into a **query**, a **key** and a **value** — three different views of itself. Our weights are random because the model is untrained; the shapes and the arithmetic are exactly a real model's.

In [ ]:
rng = np.random.default_rng(0)
d_model = 4
W_q = rng.normal(0, 0.5, size=(d_model, d_model))
W_k = rng.normal(0, 0.5, size=(d_model, d_model))
W_v = rng.normal(0, 0.5, size=(d_model, d_model))

Q = X @ W_q          # (5, 4) - one query per token
K = X @ W_k          # (5, 4) - one key per token
V = X @ W_v          # (5, 4) - one value per token
print("X:", X.shape, "  Q/K/V:", Q.shape, K.shape, V.shape)
print("query of token 1 ('cat')   :", np.round(Q[1], 3))
print("key   of token 4 ('kitten'):", np.round(K[4], 3))

Three matrices, three views — and notice that **no token has looked at any other yet**. Every row was computed independently from its own embedding. The looking happens next.

$QK^{\top}$ is an $n \times n$ table whose entry $(i, j)$ is the dot product of token $i$'s query with token $j$'s key: *how relevant token $i$ finds token $j$*.

In [ ]:
scores = Q @ K.T
print("scores:", scores.shape, "(one row per query token, one column per key token)\n")
print("      " + "".join(f"{w:>9}" for w in vocab))
for i, w in enumerate(vocab):
    print(f"{w:>6}" + "".join(f"{s:>9.3f}" for s in scores[i]))
print(f"\n{len(vocab)} tokens produced {scores.size} scores.  Double the context,")
print("quadruple this table - that is why long contexts are expensive.")

Two things happen to those scores before they become weights.

First, divide by $\sqrt{d_k}$. Dot products of $d_k$-dimensional random vectors have a standard deviation that grows like $\sqrt{d_k}$, and large numbers make softmax saturate — one token takes essentially all the weight and the gradient dies. Second, **softmax** each row, turning it into positive numbers that sum to 1:

$$
\operatorname{softmax}(z)_i = \frac{e^{z_i}}{\sum_j e^{z_j}}
$$

The cell below does both, and first shows what the scaling rescues.

In [ ]:
def softmax(v):
    e = np.exp(v - v.max())      # subtract the max: avoids overflow, same result
    return e / e.sum()

probe = np.random.default_rng(1)
print(f"{'d_k':>5}{'raw score std':>15}{'max weight unscaled':>22}{'scaled':>9}")
for dk in [4, 64, 512]:
    q = probe.normal(size=dk)
    k = probe.normal(size=(6, dk))
    raw = k @ q
    print(f"{dk:>5}{raw.std():>15.2f}{softmax(raw).max():>22.3f}"
          f"{softmax(raw / np.sqrt(dk)).max():>9.3f}")

d_k = d_model
A = np.array([softmax(row / np.sqrt(d_k)) for row in scores])   # attention weights

print("\nrow sums (must all be 1):", np.round(A.sum(axis=1), 6), "\n")
print("      " + "".join(f"{w:>9}" for w in vocab))
for i, w in enumerate(vocab):
    print(f"{w:>6}" + "".join(f"{a:>9.3f}" for a in A[i]))
print("\ncolumn sums:", np.round(A.sum(axis=0), 3))
print("'chased' pays most attention to:", vocab[int(A[2].argmax())])

At $d_k = 4$ the scaling barely matters; at $d_k = 512$ the unscaled softmax puts effectively *all* the weight on one token, a hard spike with no gradient left. One `/ np.sqrt(d_k)` is the whole difference, and forgetting it produces a model that runs fine and never learns.

Then the weights. Every row of `A` sums to 1: each token distributes exactly 100% of its attention across the sequence. The columns do **not** sum to 1 — a large column sum means many tokens are looking at that one token, which makes it an information hub.

Numbers in a grid are hard to read; a heatmap is not.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(4.6, 3.9))
im = ax.imshow(A, cmap="viridis", vmin=0, vmax=A.max())
ax.set_xticks(range(len(vocab)), vocab, rotation=45)
ax.set_yticks(range(len(vocab)), vocab)
ax.set_xlabel("attended to (keys)")
ax.set_ylabel("attending from (queries)")
ax.set_title("Attention weights (untrained, random projections)")
for i in range(len(vocab)):
    for j in range(len(vocab)):
        ax.text(j, i, f"{A[i, j]:.2f}", ha="center", va="center",
                color="white" if A[i, j] < 0.5 else "black", fontsize=8)
fig.colorbar(im, ax=ax, shrink=0.8, label="weight")
fig.tight_layout()

This is the picture you see in every interpretability paper. Ours is meaningless — random weights — but in a trained model these rows are startlingly readable: pronouns light up on the noun they refer to, closing brackets on their opener, and the final token of a sentence gathers from everything relevant.

The last step is the blend. Each token's output is the sum of every token's *value* vector, weighted by that row of `A`.

In [ ]:
out = A @ V
print("output shape:", out.shape, "(identical to the input X)")
print("output for 'chased':", np.round(out[2], 3))

manual = sum(A[2, j] * V[j] for j in range(len(vocab)))     # the same row, by hand
print("hand-computed      :", np.round(manual, 3))
print("identical:", np.allclose(out[2], manual))

That is attention, complete. Input: five vectors that knew nothing about each other. Output: five vectors, each a mixture of the whole sequence, mixed according to (here random, normally learned) relevance. Because the output shape equals the input shape, the block **stacks** — dozens of these plus a feed-forward layer each is a language model.

### Micro-exercise: the attention entropy of a row

A row of `A` that is nearly one-hot means "this token is focused"; a flat row means "this token is spreading its attention thin". Measure it with entropy, $-\sum_j a_j \log_2 a_j$, and report which token is the most focused.

In [ ]:
def row_entropy(row):
    # your code here: -sum(p * log2(p)) over the row (add 1e-12 inside the log)
    return 0.0

# Uncomment to test:
# for i, w in enumerate(vocab):
#     print(f"{w:>6}  entropy {row_entropy(A[i]):.3f} bits")
# print("flat row would be", np.log2(len(vocab)).round(3), "bits")

## 4. Causal masking: no peeking at the future

A decoder-only model is trained to predict the next token. If token 2 could see token 3, the answer would be sitting in the input and the model would learn nothing. So before the softmax we set every "future" score to $-\infty$. Since $e^{-\infty} = 0$, those weights vanish and each row renormalises over the past only.

In [ ]:
n = len(vocab)
mask = np.triu(np.ones((n, n), dtype=bool), k=1)      # True strictly above diagonal
masked_scores = np.where(mask, -np.inf, scores)
A_causal = np.array([softmax(row / np.sqrt(d_k)) for row in masked_scores])

print("masked scores, row 'chased':", np.round(masked_scores[2], 2), "\n")
print("      " + "".join(f"{w:>9}" for w in vocab))
for i, w in enumerate(vocab):
    print(f"{w:>6}" + "".join(f"{a:>9.3f}" for a in A_causal[i]))
print("\nrow sums:", np.round(A_causal.sum(axis=1), 6))
print("token 0 can only see itself:", np.round(A_causal[0], 3))
print("upper triangle is all zero:", bool(np.all(A_causal[mask] == 0)))

The result is lower-triangular: token 0 attends only to itself with weight 1.000, token 1 to tokens 0–1, and so on. This single line of masking is what makes the model **causal** — and it is also what makes the next section possible. Because token 2 never looks at token 3, token 2's key and value never change once computed, so they can be *stored and reused* instead of recalculated.

## 5. Sampling: choosing the next token

The stack ends with a vector of **logits** — one raw score per vocabulary entry. Logits are not probabilities: they are unbounded and need not sum to anything. Softmax converts them; then a *decoding strategy* picks one token.

Always taking the largest logit (greedy decoding) is reproducible and boring — and it loops. The three standard knobs reshape the distribution instead. **Temperature** $T$ divides every logit before the softmax, $p_i \propto e^{z_i / T}$: small $T$ sharpens, large $T$ flattens. Here is one distribution seen at three temperatures.

In [ ]:
SAMP_VOCAB = ["mat", "rug", "floor", "sofa", "table",
              "roof", "banana", "purple", "sideways", "the"]
logits = np.array([4.0, 3.2, 2.6, 1.8, 1.2, 0.4, -2.0, -2.5, -3.0, -3.5])

def temper(z, T):
    return softmax(z / T)

print(f"{'T':>5}{'mat':>8}{'rug':>8}{'floor':>8}{'nonsense tail':>15}{'entropy':>10}")
for T in [0.2, 0.7, 1.0, 2.0]:
    p = temper(logits, T)
    tail = float(p[6:].sum())              # the four tokens the model scored lowest
    ent = float(-(p * np.log2(p + 1e-12)).sum())
    print(f"{T:>5.1f}{p[0]:>8.3f}{p[1]:>8.3f}{p[2]:>8.3f}{tail:>15.5f}{ent:>8.2f} bits")

Watch the `nonsense tail` column: it holds the four tokens the model itself scored lowest. At $T = 0.2$ they are effectively impossible; by $T = 2.0$ they carry a real share of the mass. **High temperature does not mean "more creative"** — it means more probability on tokens the model thinks are wrong, on every single step, compounding over a paragraph.

Temperature has one bad property: it never *removes* anything. Two truncation rules do. **Top-k** keeps the $k$ highest logits and sets the rest to $-\infty$. **Top-p** (nucleus) sorts by probability, walks down accumulating mass, and stops as soon as the running total reaches $p$ — so it keeps two tokens when the model is confident and two hundred when it is not. Note in the output that `top-k=1` is exactly greedy decoding: one survivor, probability 1.000.

In [ ]:
def top_k_filter(z, k):
    if k is None or k >= len(z):
        return z.copy()
    kth = np.sort(z)[-k]                          # the k-th largest value
    return np.where(z < kth, -np.inf, z)

def top_p_filter(z, p_threshold):
    if p_threshold is None or p_threshold >= 1.0:
        return z.copy()
    probs = softmax(z)
    order = np.argsort(-probs)                    # most likely first
    cumulative = np.cumsum(probs[order])
    n_keep = int(np.searchsorted(cumulative, p_threshold) + 1)
    keep = set(order[:n_keep].tolist())
    return np.array([zi if j in keep else -np.inf for j, zi in enumerate(z)])

def survivors(z):
    return [SAMP_VOCAB[j] for j in np.argsort(-z) if np.isfinite(z[j])]

for k in [1, 3, 5]:
    print(f"top-k={k}: kept {len(survivors(top_k_filter(logits, k)))} -> "
          f"{survivors(top_k_filter(logits, k))}")
print()
for p in [0.5, 0.9, 0.99]:
    print(f"top-p={p}: kept {len(survivors(top_p_filter(logits, p)))} -> "
          f"{survivors(top_p_filter(logits, p))}")

# ---- all five settings over the same logits, on one axis ----------------
xs = np.arange(len(SAMP_VOCAB))
settings = [("T=1.0 (raw)", softmax(logits)),
            ("T=0.5 (sharper)", temper(logits, 0.5)),
            ("T=1.5 (flatter)", temper(logits, 1.5)),
            ("top-k=3", softmax(top_k_filter(logits, 3))),
            ("top-p=0.9", softmax(top_p_filter(logits, 0.9)))]

fig, ax = plt.subplots(figsize=(8.2, 3.6))
width = 0.16
for offset, (label, p) in zip(np.linspace(-2, 2, len(settings)) * width, settings):
    ax.bar(xs + offset, p, width=width, label=label)
ax.set_xticks(xs, SAMP_VOCAB, rotation=45, ha="right")
ax.set_xlabel("candidate next token")
ax.set_ylabel("probability")
ax.set_title("One set of logits, five decoding settings")
ax.legend(fontsize=8)
fig.tight_layout()

for label, p in settings:
    print(f"{label:<18} support={int((p > 1e-12).sum()):>2} tokens   "
          f"entropy={-(p * np.log2(p + 1e-12)).sum():.2f} bits")

The two truncation bars are visibly different from the temperature bars: they are *zero* outside their support, not merely small. That is the point — a token that has been filtered out cannot be sampled, no matter how unlucky your random draw is. Typical production defaults pair a moderate temperature with top-p around 0.9–0.95, sometimes with a generous top-k as a safety net.

### Micro-exercise: sample and count

Write `sample_counts(p, n, seed)` that draws `n` tokens from distribution `p` with `np.random.default_rng(seed)` and returns a `Counter` of token names. Compare the empirical frequencies at $T = 0.5$ and $T = 1.5$ against the probabilities printed above.

In [ ]:
from collections import Counter

def sample_counts(p, n=2000, seed=0):
    rng = np.random.default_rng(seed)
    # your code here: rng.choice over indices with p=p, then count the names
    return Counter()

# Uncomment to test:
# print(sample_counts(temper(logits, 0.5)).most_common(3))
# print(sample_counts(temper(logits, 1.5)).most_common(3))

## 6. The KV cache: counting the work you can skip

Generation is a loop: run the model over the conversation so far, pick a token, append it, repeat. Written naively, step 400 recomputes almost everything step 399 computed — because causal masking means an old token's key and value **can never change**.

Two quantities matter. How many token vectors get pushed through the projection matrices, and how many query-key dot products get computed. Count them.

In [ ]:
def work(prompt_len, n_new):
    """Count both kinds of work for generating n_new tokens."""
    no_cache_proj = no_cache_scores = cached_proj = cached_scores = 0
    for step in range(n_new):
        s = prompt_len + step                  # tokens already in the sequence
        no_cache_proj += s                     # re-project the whole prefix
        no_cache_scores += s * (s + 1) // 2    # causal: query i sees i+1 keys
        cached_proj += 1                       # one new token in
        cached_scores += s                     # one query against s stored keys
    return no_cache_proj, cached_proj, no_cache_scores, cached_scores

head = (f"{'new':>6} | {'proj none':>11} {'cached':>7} {'save':>7} | "
        f"{'scores none':>13} {'cached':>9} {'save':>7}")
print(head)
print("-" * len(head))
for n in [8, 32, 128, 512, 2048]:
    p_no, p_yes, s_no, s_yes = work(prompt_len=1, n_new=n)
    print(f"{n:>6} | {p_no:>11,} {p_yes:>7,} {p_no / p_yes:>6.1f}x | "
          f"{s_no:>13,} {s_yes:>9,} {s_no / s_yes:>6.1f}x")

Two separate Big-O stories in one table.

**Projections.** Without a cache the count is $1 + 2 + \dots + n = \Theta(n^2)$ — the same triangular sum that makes bubble sort quadratic. With a cache it is exactly $n$. This is the expensive half, because projections are multiplications by the model's big weight matrices, and the saving grows without bound.

**Attention scores.** Without a cache each step recomputes the whole causal triangle, $\Theta(n^3)$ overall. With a cache it is $\Theta(n^2)$ overall. That remaining quadratic is *irreducible* — attention genuinely looks at every earlier token. The cache removes the repeated work, not the algorithm's cost. And note what the cache does **not** change: the output. It is pure memoization of numbers that are identical either way.

The bill arrives as memory. Per token the cache stores K and V (the 2), for every layer $L$, for every key/value head $H_{kv}$, each of width $d_{head}$, at $b$ bytes per number:

$$
\text{KV bytes} = 2 \times L \times H_{kv} \times d_{head} \times n_{tokens} \times b
$$

In [ ]:
GB = 1e9                                  # memory reported with GB = 10**9 bytes

def kv_bytes(layers, kv_heads, head_dim, n_tokens, bytes_per_elt=2):
    """2 (K and V) x layers x KV heads x head dim x tokens x bytes."""
    return 2 * layers * kv_heads * head_dim * n_tokens * bytes_per_elt

# (label, layers, KV heads, head_dim) -- shapes of widely deployed open models.
CONFIGS = [
    ("7B   MHA   (32 L, 32 KV heads)", 32, 32, 128),
    ("8B   GQA-8 (32 L,  8 KV heads)", 32, 8, 128),
    ("7B   MQA-1 (32 L,  1 KV head )", 32, 1, 128),
    ("70B  MHA   (80 L, 64 KV heads)", 80, 64, 128),
    ("70B  GQA-8 (80 L,  8 KV heads)", 80, 8, 128),
]

head = (f"{'configuration (fp16)':<32} {'MB/token':>9} {'4k ctx':>9} "
        f"{'32k ctx':>9} {'128k ctx':>10} {'reqs in 40GB':>13}")
print(head)
print("-" * len(head))
for name, L, H_kv, d_head in CONFIGS:
    per_token = kv_bytes(L, H_kv, d_head, 1)
    sizes = [kv_bytes(L, H_kv, d_head, ctx) / GB for ctx in (4096, 32768, 131072)]
    fits = int(40 * GB / kv_bytes(L, H_kv, d_head, 4096))
    print(f"{name:<32} {per_token / 1e6:>8.3f}  {sizes[0]:>8.2f}  {sizes[1]:>8.2f}  "
          f"{sizes[2]:>9.2f}  {fits:>12}")

mha, gqa = kv_bytes(80, 64, 128, 4096), kv_bytes(80, 8, 128, 4096)
print(f"\n70B at 4k tokens: GQA-8 needs {mha / gqa:.0f}x less KV memory than MHA "
      f"({mha / GB:.1f} GB -> {gqa / GB:.1f} GB)")

This table is the "why your GPU ran out of memory" moment, and the one term worth staring at is $H_{kv}$. Classic **multi-head attention** gives every query head its own key/value head. **Grouped-query attention** (GQA) lets several query heads share one KV head; **multi-query attention** (MQA) takes it to the limit with a single shared KV head. The query heads are unchanged — only the cache shrinks, linearly. That is why nearly every model released for serving uses GQA.

To size your own model, open its `config.json` and read `num_hidden_layers`, `num_key_value_heads`, and $d_{head} = \text{hidden\_size} / \text{num\_attention\_heads}$. Then multiply by the number of concurrent requests you intend to serve, *before* choosing a maximum context length.

## What you built

| Piece | Lines of numpy | The idea in one sentence |
|---|---|---|
| BPE | ~35 | Glue the most frequent adjacent pair, repeatedly; keep the recipe. |
| Embedding | 1 | A token's vector is a row lookup. |
| Attention | ~9 | Score every pair with a dot product, softmax the rows, blend the values. |
| Causal mask | 2 | Set future scores to $-\infty$ so a token cannot read its own answer. |
| Sampling | ~12 | Reshape the logits (temperature), or delete part of them (top-k / top-p). |
| KV cache | ~6 | Old keys and values cannot change, so store them instead of recomputing. |

## Try it yourself

Bigger exercises. Every scaffold runs as-is.

### Exercise 1 — Merge count versus token count

Train BPE on `CORPUS` with 0, 5, 10, 20 and 40 merges, and for each report the vocabulary size and the number of tokens `encode(CORPUS)` produces. Plot tokens against vocabulary size, labelling both axes. You are drawing the central tokenizer trade-off: a bigger vocabulary means shorter sequences and a bigger embedding matrix.

In [ ]:
def vocab_and_tokens(n_merges):
    ms = train_bpe(CORPUS, n_merges)
    r = {p: i for i, (p, _c) in enumerate(ms)}

    def enc_word(word):
        syms = list(symbols(word))
        while True:
            opts = [(r[p], i) for i, p in enumerate(zip(syms, syms[1:])) if p in r]
            if not opts:
                return syms
            _, i = min(opts)
            syms[i:i + 2] = [syms[i] + syms[i + 1]]

    base_syms = {c for w in CORPUS.lower().split() for c in symbols(w)}
    size = len(base_syms | {a + b for (a, b), _c in ms})
    n_tok = sum(len(enc_word(w)) for w in CORPUS.lower().split())
    return size, n_tok

# your code here: loop over [0, 5, 10, 20, 40], collect the pairs, plot them
for m in [0, 5, 10, 20, 40]:
    print(m, "merges ->", vocab_and_tokens(m))

### Exercise 2 — A second attention head that finds the verb

Multi-head attention runs several attention computations in parallel, each in $d_{\text{model}}/h$ dimensions, then concatenates and mixes them. Hand-build a head whose query matrix makes *every* token ask for the `action` dimension (column 1) and whose key matrix advertises only that dimension. Print which token every row attends to most — it should be `chased` for every token allowed to see it.

In [ ]:
d_head = d_model // 2
Wq_verb = np.array([[0.0, 1.0]] * d_model)        # every query asks for column 1
Wk_verb = np.array([[0.0, 0.0], [0.0, 1.0], [0.0, 0.0], [0.0, 0.0]])
Wv_verb = rng.normal(0, 0.5, size=(d_model, d_head))

def one_head(Xin, Wq, Wk, Wv, causal=True):
    q, k, v = Xin @ Wq, Xin @ Wk, Xin @ Wv
    s = q @ k.T / np.sqrt(Wq.shape[1])
    if causal:
        s = np.where(mask, -np.inf, s)
    weights = np.array([softmax(r) for r in s])
    return weights, weights @ v

# your code here: call one_head, then print the argmax token of each row

### Exercise 3 — Where does temperature stop being safe?

For temperatures from 0.1 to 3.0, compute the total probability that `top_p_filter` at $p = 0.9$ would *discard*, and the probability mass sitting on the four nonsense tokens (indices 6–9). Plot both against temperature. Find, numerically, the temperature at which the nonsense mass first exceeds 5%.

In [ ]:
temps = np.linspace(0.1, 3.0, 30)
nonsense = []
# your code here: for each T, p = temper(logits, T); append float(p[6:].sum())

# Uncomment once nonsense is filled in:
# fig, ax = plt.subplots(figsize=(6.4, 3.2))
# ax.plot(temps, nonsense)
# ax.axhline(0.05, ls="--", color="0.6")
# ax.set_xlabel("temperature T")
# ax.set_ylabel("probability mass on nonsense tokens")
# ax.set_title("When sampling starts producing junk")
# fig.tight_layout()

### Exercise 4 — Your own KV budget

Pick a context length and a number of concurrent users, then write `max_context(gpu_gb, n_users, layers, kv_heads, head_dim)` returning the longest context you can serve without exceeding the KV budget. Check it against the 70B GQA-8 row of the table: how many tokens fit for 16 users in 40 GB? Then re-run it in 8-bit (`bytes_per_elt=1`) and explain the factor you see.

In [ ]:
def max_context(gpu_gb, n_users, layers, kv_heads, head_dim, bytes_per_elt=2):
    per_token_per_user = kv_bytes(layers, kv_heads, head_dim, 1, bytes_per_elt)
    # your code here: return the integer number of tokens per user that fits
    return 0

# Uncomment to test:
# print("70B GQA-8, 16 users, 40 GB, fp16:", max_context(40, 16, 80, 8, 128))
# print("70B GQA-8, 16 users, 40 GB, int8:", max_context(40, 16, 80, 8, 128, 1))